# Validate published PLUMED conda packages on a Colab GPU

After merging the packaging PR, run CI on master with **conda_test_upload** enabled
and wait for publication to finish. Then select **Runtime → Change runtime type → GPU**
and **Run all** here. No laptop uploads or GitHub token are needed.

The notebook installs from **plumed/label/test**, uses the source revision recorded
inside that exact kernel package, and derives the CUDA/OpenACC alternatives from
its module declarations. Leave `requested_version` empty for the newest snapshot,
or enter a published date such as `2026.11.13` to repeat an older one.

Each variant runs the maintained GPU numerical tests with automatic and explicit
loading. PASS requires every declared alternative to pass. Driver compatibility
is never overridden, and CPU fallback cannot satisfy the OpenACC GPU checks.

The final cell downloads **plumed-gpu-results.zip**, containing logs, package
versions/checksums, source provenance and results. Environments and binaries are
excluded. There are no expiring CI artifact links or version matrices to refresh.

In [ ]:
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

channel = "plumed/label/test"
requested_version = None  # Optional published date; otherwise select the newest.
work_root = Path("/content/plumed-gpu-validation")
setup_logs = work_root / "logs"
setup_logs.mkdir(parents=True, exist_ok=True)
conda_prefix = work_root / "conda"
source_dir = work_root / "source"
run_dir = work_root / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
run_dir.mkdir(parents=True)
kernel_prefix = run_dir / "kernel"
validation_env = dict(os.environ)
for key in list(validation_env):
    if key.startswith("PLUMED_") or key in (
        "CONDA_OVERRIDE_CUDA",
        "PYTHONPATH",
        "PYTHONHOME",
    ):
        validation_env.pop(key)
validation_env.update(
    CONDA_PKGS_DIRS=str(work_root / "package-cache"),
    CONDA_REGISTER_ENVS="false",
    CONDA_NUMBER_CHANNEL_NOTICES="0",
)


def download(url, destination, expected_sha256=None):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if (
        expected_sha256
        and destination.exists()
        and hashlib.sha256(destination.read_bytes()).hexdigest() == expected_sha256
    ):
        return destination
    for attempt in range(3):
        try:
            request = urllib.request.Request(
                url, headers={"User-Agent": "PLUMED-Colab-validation"}
            )
            with (
                urllib.request.urlopen(request, timeout=60) as response,
                destination.with_suffix(destination.suffix + ".part").open(
                    "wb"
                ) as output,
            ):
                shutil.copyfileobj(response, output)
            destination.with_suffix(destination.suffix + ".part").replace(destination)
            break
        except OSError as error:
            if attempt == 2:
                raise RuntimeError(
                    f"Download failed: {url}: {error}. Retry the cell. Check the network connection and retry."
                ) from None
            time.sleep(2**attempt)
    if expected_sha256:
        actual = hashlib.sha256(destination.read_bytes()).hexdigest()
        if actual != expected_sha256:
            raise RuntimeError(f"Checksum mismatch for {destination.name}: {actual}")
    return destination


def run_logged(command, log_path, *, env=None, cwd=None, check=True):
    print("Running:", " ".join(map(str, command)), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w") as output:
        result = subprocess.run(
            list(map(str, command)),
            env=env or validation_env,
            cwd=cwd,
            stdout=output,
            stderr=subprocess.STDOUT,
            check=False,
        )
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-20:]))
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed; see {log_path.name}. Run the final cell to download available logs."
        )
    return result.returncode

In [ ]:
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError(
        "Use a hosted Linux x86-64 Colab GPU runtime for these packages."
    )
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime is connected. Select a GPU runtime and rerun."
    )
run_logged(["nvidia-smi"], setup_logs / "hardware.log")

In [ ]:
# Bootstrap the current official Miniforge release and verify its published checksum.
conda = conda_prefix / "bin/conda"
if not conda.exists():
    release_path = download(
        "https://api.github.com/repos/conda-forge/miniforge/releases/latest",
        setup_logs / "miniforge-release.json",
    )
    release = json.loads(release_path.read_text())
    assets = {item["name"]: item["browser_download_url"] for item in release["assets"]}
    installer_name = next(
        name
        for name in assets
        if name.startswith("Miniforge3-")
        and name.endswith("-Linux-x86_64.sh")
        and name + ".sha256" in assets
    )
    checksum_file = download(
        assets[installer_name + ".sha256"], work_root / "downloads/miniforge.sha256"
    )
    checksum = checksum_file.read_text().split()[0]
    if not re.fullmatch(r"[0-9a-f]{64}", checksum):
        raise RuntimeError("Invalid Miniforge checksum response")
    installer = download(
        assets[installer_name], work_root / "downloads/miniforge.sh", checksum
    )
    run_logged(
        ["bash", installer, "-b", "-p", conda_prefix],
        setup_logs / "conda-bootstrap.log",
    )
validation_env["PATH"] = str(conda_prefix / "bin") + os.pathsep + os.environ["PATH"]
base_spec = "plumed" + ("=" + requested_version if requested_version else "")
run_logged(
    [
        conda,
        "create",
        "-y",
        "-p",
        kernel_prefix,
        "--strict-channel-priority",
        "--override-channels",
        "-c",
        channel,
        "-c",
        "conda-forge",
        base_spec,
    ],
    setup_logs / "kernel-install.log",
)
kernel_record = next(
    record
    for path in (kernel_prefix / "conda-meta").glob("plumed-*.json")
    if (record := json.loads(path.read_text()))["name"] == "plumed"
)
version = kernel_record["version"]
kernel_spec = "plumed=" + version + "=" + kernel_record["build"]
provenance_path = kernel_prefix / "lib/plumed/conda-source.json"
if not provenance_path.exists():
    raise RuntimeError(
        "This older package has no source provenance. Publish a new snapshot from master first."
    )
provenance = json.loads(provenance_path.read_text())
source_revision, repository = provenance.get("revision"), provenance.get("repository")
if (
    not isinstance(source_revision, str)
    or not re.fullmatch(r"[0-9a-f]{40}", source_revision)
    or provenance.get("dirty") is not False
):
    raise RuntimeError(
        "The package does not identify a clean, reproducible source revision."
    )
if not isinstance(repository, str) or not re.fullmatch(
    r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository
):
    raise RuntimeError("Invalid source repository in package provenance")
(setup_logs / "package-source.json").write_text(
    json.dumps(
        {"channel": channel, "kernel": kernel_record, "source": provenance}, indent=2
    )
    + "\n"
)
print("Selected:", kernel_spec, "source:", repository, source_revision)

In [ ]:
# Fetch the test harness and fixtures belonging to the installed package.
if not (source_dir / ".git").exists():
    source_dir.mkdir(parents=True, exist_ok=True)
    run_logged(["git", "init", source_dir], setup_logs / "source-init.log")
    run_logged(
        [
            "git",
            "-C",
            source_dir,
            "remote",
            "add",
            "origin",
            "https://github.com/" + repository + ".git",
        ],
        setup_logs / "source-remote.log",
    )
else:
    run_logged(
        [
            "git",
            "-C",
            source_dir,
            "remote",
            "set-url",
            "origin",
            "https://github.com/" + repository + ".git",
        ],
        setup_logs / "source-remote.log",
    )
run_logged(
    [
        "git",
        "-C",
        source_dir,
        "fetch",
        "--depth=1",
        "--filter=blob:none",
        "origin",
        source_revision,
    ],
    setup_logs / "source-download.log",
)
run_logged(
    ["git", "-C", source_dir, "checkout", "--detach", "--force", source_revision],
    setup_logs / "source-checkout.log",
)
print("Setup complete. The next cell executes the GPU numerical tests.")

In [ ]:
# The maintained source harness selects variants and rejects numerical skips.
test_env = dict(
    validation_env,
    VERSION=version,
    GPU_KERNEL_SPEC=kernel_spec,
    GPU_CONDA_CHANNEL=channel,
    GPU_TEST_ROOT=str(run_dir),
)
exit_code = run_logged(
    ["bash", source_dir / "conda/run-gpu-test.sh"],
    run_dir / "runner.log",
    env=test_env,
    check=False,
)
summary_path = run_dir / "summary.json"
summary = (
    json.loads(summary_path.read_text())
    if summary_path.exists()
    else {
        "passed": False,
        "status": "GPU run interrupted or incomplete",
        "variants": {},
    }
)
summary.update(source=provenance, kernel=kernel_spec, exit_code=exit_code)
summary["passed"] = bool(summary.get("passed")) and exit_code == 0
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
for variant, result in summary["variants"].items():
    print(variant, result["status"])
print(
    "OVERALL:",
    "PASS" if summary["passed"] else "INCOMPLETE / FAILED — return the logs for review",
)

### Return the results

Run the final cell even if setup or testing failed. It downloads the available
logs, source provenance, package records and results. A driver incompatibility is
an installation/test failure and does not count as successful validation.

In [ ]:
from google.colab import files

if run_dir is not None and not (run_dir / "summary.json").exists():
    (run_dir / "summary.json").write_text(
        json.dumps(
            {"passed": False, "status": "GPU run interrupted or incomplete"}, indent=2
        )
        + "\n"
    )
result_zip = work_root / "plumed-gpu-results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(setup_logs.glob("*")):
        if path.is_file() and path.suffix in (".log", ".json"):
            archive.write(path, "setup/" + path.name)
    if run_dir is not None:
        for path in sorted(run_dir.glob("*")):
            if path.is_file() and path.suffix in (".log", ".json"):
                archive.write(path, "tests/" + path.name)
print(
    f"Results: {result_zip.stat().st_size:,} bytes; environments and binaries excluded."
)
if run_dir is None:
    print("GPU tests have not run; this ZIP contains setup diagnostics only.")
files.download(str(result_zip))